In [0]:
-- =============================================================================
-- HEALTHCARE DLT PIPELINE - BRONZE LAYER: DIAGNOSTIC MAPPING
-- =============================================================================
-- This table creates a bronze layer for diagnosis code mappings
-- Purpose: Reference data for diagnosis codes to descriptions
-- Data Quality: Enforces non-null constraints on critical fields
-- Violation Handling: Drops rows with null diagnosis codes or descriptions

create live table diagnostic_mapping( 
-- Data Quality Constraints: Ensure critical fields are not null
constraint dg_code_not_null expect (diagnosis_code is not null) on violation drop row,
constraint dg_description_not_null expect (diagnosis_description is not null) on violation drop row 
)
comment "Bronze table for the diagnosis mapping file - Reference data for diagnosis codes"
TBLPROPERTIES ("quality" = "bronze")
as
select 
-- Explicit type casting for data consistency
cast(diagnosis_code as string) diagnosis_code, 
cast(diagnosis_description as string) diagnosis_description 
from healthcare.default.raw_diagnosis_map

In [0]:
-- =================================================================
-- HEALTHCARE DLT PIPELINE - BRONZE LAYER: DAILY PATIENTS (STREAMING)
-- =============================================================================
-- This table creates a streaming bronze layer for daily patient admissions
-- Purpose: Real-time ingestion of patient data with comprehensive data quality checks
-- Streaming: Uses STREAM() to process new data as it arrives
-- Data Quality: Enforces business rules for patient data completeness

create or refresh streaming table daily_patients (
-- Primary Key Constraint: Patient ID must be present
constraint pk_not_null expect (patient_id is not null) on violation drop row,
-- Business Rule: All essential patient fields must be populated
constraint required_fields expect (name is not null and age is not null and 
gender is not null and address is not null and contact_number is not null and 
admission_date is not null) on violation drop row 
)
comment "Bronze table for daily patient data - Streaming ingestion with data quality enforcement"
TBLPROPERTIES ("quality" = "bronze")
as 
select 
-- Explicit type casting for data consistency and validation
cast(patient_id as string) patient_id, cast(name as string) name,
cast(age as int) age, cast(gender as string) gender, 
cast(address as string) address, cast(contact_number as string) contact_number, 
cast(admission_date as date) admission_date, cast(diagnosis_code as string) diagnosis_code 
from stream("healthcare.default.raw_patients_daily")

In [0]:
-- =============================================================================
-- HEALTHCARE DLT PIPELINE - SILVER LAYER: PROCESSED PATIENT DATA
-- =============================================================================
-- This table creates a silver layer by joining patient data with diagnosis mappings
-- Purpose: Enriched patient data with human-readable diagnosis descriptions
-- Data Quality: Ensures diagnosis descriptions are available for analysis
-- Join Strategy: LEFT JOIN to preserve all patients, even those with unmapped codes

create or refresh streaming table processed_patient_data (
-- Data Quality: Ensure diagnosis description is available for meaningful analysis
constraints has_diagnosis expect (diagnosis_description is not null) on violation drop row
)
comment "Silver table with enriched patient data - Joined with diagnosis mappings for analysis"
TBLPROPERTIES("quality" = "silver")
as 
select p.patient_id, p.name, p.age, p.gender, p.address, p.contact_number,
p.admission_date, m.diagnostic_description as diagnosis_description 
from stream(live.daily_patients) p left join live.diagnostic_mapping m 
on p.diagnosis_code=m.diagnosis_code

In [0]:
-- =============================================================================
-- HEALTHCARE DLT PIPELINE - GOLD LAYER: PATIENT STATISTICS BY ADMISSION DATE
-- =============================================================================
-- This table creates aggregated analytics for daily patient admissions by diagnosis
-- Purpose: Daily operational metrics for hospital capacity and diagnosis trends
-- Aggregation: Groups by admission date and diagnosis for time-series analysis
-- Metrics: Patient counts and average age for demographic insights

create live table patient_statistics_by_admission_date 
comment "Gold table with daily patient admission statistics by diagnosis - Operational metrics"
TBLPROPERTIES ("quality" = "gold")
as 
select 
-- Time dimension for trend analysis
admission_date, 
-- Diagnosis dimension for medical insights
diagnosis_description, 
-- Key operational metrics
count(patient_id) patient_count, 
avg(age) avg_age
from live.processed_patient_data 
group by admission_date, diagnosis_description

In [0]:
-- =============================================================================
-- HEALTHCARE DLT PIPELINE - GOLD LAYER: PATIENT STATISTICS BY DIAGNOSIS
-- =============================================================================
-- This table creates comprehensive analytics for each diagnosis type
-- Purpose: Medical insights and demographic analysis by diagnosis
-- Aggregation: Groups by diagnosis for medical research and capacity planning
-- Metrics: Patient counts, age statistics, and gender distribution

create live table patient_statistics_by_diagnosis 
comment "Gold table with comprehensive patient statistics by diagnosis - Medical analytics"
TBLPROPERTIES ("quality"="gold")
as 
select 
-- Diagnosis dimension for medical categorization
diagnosis_description, 
-- Patient volume metrics
count(patient_id) patient_count, 
-- Age demographic analysis
avg(age) avg_age,
min(age) min_age, max(age) max_age,
-- Gender distribution analysis
count(distinct gender) unique_gender_count,
from live.processed_patient_data 
group by diagnosis_description;

In [0]:
-- =============================================================================
-- HEALTHCARE DLT PIPELINE - GOLD LAYER: PATIENT STATISTICS BY GENDER
-- =============================================================================
-- This table creates demographic analytics grouped by patient gender
-- Purpose: Gender-based health insights and demographic analysis
-- Aggregation: Groups by gender for population health studies
-- Metrics: Patient counts, age statistics, and diagnosis diversity

CREATE LIVE TABLE patient_statistics_by_gender
COMMENT "Gold table with demographic patient statistics by gender - Population health analytics"
TBLPROPERTIES ("quality" = "gold")
AS
SELECT
    -- Gender dimension for demographic analysis
    gender,
    -- Patient volume metrics
    COUNT(patient_id) AS patient_count,
    -- Age demographic analysis
    AVG(age) AS avg_age,
    MIN(age) AS min_age,
    MAX(age) AS max_age,
    -- Medical diversity analysis
    COUNT(DISTINCT diagnosis_description) AS unique_diagnosis_count
FROM live.processed_patient_data
GROUP BY gender;